# Level 3: Core Numerical Methods Engine
**Goal:** Implement and compare core numerical methods from scratch to support irrigation decisions.

### Mathematical Context
We use the discrete water-balance equation:
$$S_{t+1} = S_t + R_t + I_t - ET_t - D_t$$
To find the required irrigation ($I_t$), we solve for the root of:
$$f(I_t) = (S_t + R_t + I_t - ET_t - D_t) - S_{target} = 0$$

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Ensure src is in the path
sys.path.append(os.path.abspath('../src'))
import numerical_methods as nm

# 1. Load crop parameters for target moisture
params_df = pd.read_csv('../data/raw/crop_zone_parameters.csv')
zone_a_target = params_df[params_df['zone_id'] == 'Zone A']['target_moisture_pct'].values[0]

print(f"Target Moisture for Zone A: {zone_a_target}%")

Target Moisture for Zone A: 33%


## 1. Root Finding: Calculating Required Irrigation
We compare Bisection, Newton-Raphson, and Secant methods to find the exact $I_t$ needed.

In [2]:
# Define the physical state
s_t = 28.5    # Current moisture
r_t = 0.0     # No rain
et_t = 4.2    # Estimated ET
d_t = 0.5     # Drainage
target = zone_a_target

# f(I) = s_t + r_t + I - et_t - d_t - target
def f(I): return (s_t + r_t + I - et_t - d_t) - target
def df(I): return 1.0  # Derivative of f(I) with respect to I is 1

# Solve using 3 methods
root_bisect = nm.bisection_method(f, 0, 50)
root_newton = nm.newton_raphson(f, df, x0=10)
root_secant = nm.secant_method(f, x0=0, x1=50)

print(f"Bisection Result: {root_bisect:.4f} L")
print(f"Newton-Raphson Result: {root_newton:.4f} L")
print(f"Secant Result: {root_secant:.4f} L")

Bisection Result: 9.2000 L
Newton-Raphson Result: 9.2000 L
Secant Result: 9.2000 L


### Deliverable: Convergence Comparison Table
We manually track the approximate accuracy and "cost" (iterations) of each method.

In [3]:
comparison_data = {
    "Method": ["Bisection", "Newton-Raphson", "Secant"],
    "Result (I_t)": [root_bisect, root_newton, root_secant],
    "Calculated f(I)": [f(root_bisect), f(root_newton), f(root_secant)],
    "Convergence Status": ["Converged", "Converged", "Converged"]
}
comparison_table = pd.DataFrame(comparison_data)
display(comparison_table)

,Method,Result (I_t),Calculated f(I),Convergence Status
0,Bisection,9.200001,7.629395e-07,Converged
1,Newton-Raphson,9.200000,0.000000e+00,Converged
2,Secant,9.200000,0.000000e+00,Converged


## 2. Numerical Differentiation: Rate of Moisture Change
We analyze how fast soil moisture is dropping using finite differences.

In [4]:
# Synthetic moisture curve over 5 days: S(t) = 35 - 2*t - 0.1*t^2
def S_curve(t): return 35 - 2*t - 0.1*(t**2)

day = 2.0
forward_rate = nm.forward_difference(S_curve, day)
central_rate = nm.central_difference(S_curve, day)

print(f"Moisture change rate at Day 2 (Forward): {forward_rate:.4f} %/day")
print(f"Moisture change rate at Day 2 (Central): {central_rate:.4f} %/day")

Moisture change rate at Day 2 (Forward): -2.4000 %/day
Moisture change rate at Day 2 (Central): -2.4000 %/day


## 3. Numerical Integration: Cumulative Water Deficit
We integrate the moisture deficit over a 7-day period to find the total volume of water needed.

In [7]:
# Deficit function: Target (33%) - Current(t)
def deficit_func(t): return 33 - (35 - 2.5*t)

total_deficit_trap = nm.trapezoidal_rule(deficit_func, 0, 7, n=100)
total_deficit_simp = nm.simpson_rule(deficit_func, 0, 7, n=100)

delta = abs(total_deficit_trap - total_deficit_simp)

integration_data = {
    "Rule": ["Trapezoidal", "Simpson's"],
    "Estimated Total Deficit": [total_deficit_trap, total_deficit_simp],
    "Delta": [delta, delta]
}
display(pd.DataFrame(integration_data))

,Rule,Estimated Total Deficit,Delta
0,Trapezoidal,47.25,1.421085e-14
1,Simpson's,47.25,1.421085e-14


## 4. Linear Systems: 3-Zone Water Allocation
We solve for $x$ (liters per zone) in $Ax = b$, where $A$ represents system pressure/resistance and $b$ is the total required volume.

In [6]:
# A: Interaction matrix between Zone A, B, and C flow valves
# b: Target total moisture volume required for each zone
A = np.array([
    [1.5, 0.2, 0.1],
    [0.3, 1.2, 0.2],
    [0.1, 0.3, 1.1]
])
b = np.array([45.0, 38.0, 29.0])

allocation = nm.gaussian_elimination(A, b)

print("Allocated water volume (Liters):")
for zone, vol in zip(['Zone A', 'Zone B', 'Zone C'], allocation):
    print(f"{zone}: {vol:.2f} L")

Allocated water volume (Liters):
Zone A: 25.84 L
Zone B: 22.21 L
Zone C: 17.96 L


"The convergence and integration results are consistent across methods because the underlying physical models are currently linear. This verifies that our manual implementations of Bisection, Newton, and Simpson's Rule are) were different, the Gaussian solver accounted for the pipe interference.
* To get 45 units of "effective" water in Zone A, the pump actually needs to push 25.84 Liters into mathematically sound before we move to more complex, non-linear simulations in Level 5."